# モデル評価
学習済みモデルの評価指標・特徴量重要度・回収率シミュレーションを確認する

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yaml

with open('../config.yaml') as f:
    config = yaml.safe_load(f)

from src.features.feature_pipeline import load_features, FEATURE_COLUMNS
from src.model.predictor import Predictor
from src.model.evaluator import evaluate
from src.model.model_store import ModelStore

df = load_features(config)
df['race_date'] = pd.to_datetime(df['race_date'])
test_year = config['model']['test_year']
test_df = df[df['race_date'].dt.year == test_year].copy()
print(f'Test data: {len(test_df)} entries, {test_df["race_id"].nunique()} races')

In [ ]:
# 予測実行
predictor = Predictor(config)
pred_df = predictor.predict_batch(test_df)

# 評価指標
metrics = evaluate(pred_df)
print('\n【評価指標】')
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k}: {v:.4f}')
    else:
        print(f'  {k}: {v}')

In [ ]:
# 特徴量重要度
from src.model.trainer import Trainer
store = ModelStore(config)
import lightgbm as lgb
model = store.load()

importance = pd.DataFrame({
    'feature': FEATURE_COLUMNS,
    'importance': model.feature_importance(importance_type='gain')
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(8, 10))
importance.plot(kind='barh', x='feature', y='importance', ax=ax, legend=False)
ax.set_title('特徴量重要度 (Gain)')
plt.tight_layout()
plt.show()

In [ ]:
# 人気別の的中率（モデル vs 市場）
pred_df['is_win'] = (pred_df['finish'] == 1).astype(int)
pred_df['model_top1'] = pred_df.groupby('race_id')['win_prob'].rank(ascending=False) == 1
pred_df['market_top1'] = pred_df['popularity'] == 1

model_acc = pred_df[pred_df['model_top1']]['is_win'].mean()
market_acc = pred_df[pred_df['market_top1']]['is_win'].mean()

print(f'モデル1位予測の勝率: {model_acc:.1%}')
print(f'市場1番人気の勝率:   {market_acc:.1%}')